# FaceSense AI — Model Training & Continuous Retraining
> **Phase 11: Feedback-Based Retraining, Model Comparison, and Safe Promotion**

This notebook serves as the **official interactive experimentation and orchestration interface** for FaceSense AI.
Core training, dataset mixing, evaluation, and promotion logic reside in modular Python packages under `ml/`.

### Architecture Principle
- **Notebook**: Experiment orchestration, transparent parameter configuration, step-by-step visibility, visualizations, and metric analysis.
- **Python Modules (`ml/`)**: Reusable, tested implementation logic (no duplicated training loops or business logic in the notebook).

---
## 01 · Experiment Objective

### Goal
Train a candidate facial expression recognition model using the original FER2013 dataset plus validated user feedback samples, evaluate it fairly on the untouched test set, compare it against the current production model, and decide whether to promote it.

### Key Experiment Questions
1. Does incorporating human feedback improve generalization on hard or underrepresented emotion classes?
2. Does the candidate model maintain or improve overall Macro F1 score on the untouched FER2013 test set?
3. Does the candidate pass all promotion gate safety thresholds without regressing on any single class?

---
## 02 · Previous Production Baseline

Before configuring or running any training, we inspect the current **production model** and its historical benchmark performance.
This establishes the exact baseline that the candidate model will be measured against.

In [1]:
# ── Section 02: Production Baseline Inspection ─────────────────────────────
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROD_CKPT_PATH = PROJECT_ROOT / "ml" / "models" / "checkpoints" / "final" / "best_model.pt"
PROD_META_PATH = PROJECT_ROOT / "ml" / "models" / "checkpoints" / "final" / "model_metadata.json"

print("=" * 68)
print("  CURRENT PRODUCTION MODEL BASELINE")
print("=" * 68)
print(f"  Checkpoint Path : {PROD_CKPT_PATH}")
print(f"  Exists on disk  : {PROD_CKPT_PATH.exists()}")

if PROD_META_PATH.exists():
    with open(PROD_META_PATH, "r", encoding="utf-8") as f:
        prod_meta = json.load(f)
    print(f"  Architecture    : {prod_meta.get('model_architecture', 'Unknown')}")
    print(f"  Trained Epoch   : {prod_meta.get('best_epoch', 'Unknown')}")
    print(f"  Total Parameters: {prod_meta.get('total_parameters', 0):,}")
    print(f"  Test Accuracy   : {prod_meta.get('test_accuracy', 0.0):.2f}%")
    print(f"  Test Macro F1   : {prod_meta.get('test_macro_f1', 0.0):.2f}%")
    print(f"  Test Weighted F1: {prod_meta.get('test_weighted_f1', 0.0):.2f}%")
    print(f"  Test Loss       : {prod_meta.get('test_loss', 0.0):.4f}")
else:
    print("  Metadata file not found; baseline metrics will be evaluated directly.")
print("=" * 68)


  CURRENT PRODUCTION MODEL BASELINE
  Checkpoint Path : c:\Users\moham\OneDrive\Desktop\AI projects\AI Emotion Detection\ml\models\checkpoints\final\best_model.pt
  Exists on disk  : True
  Architecture    : ResidualEmotionCNN
  Trained Epoch   : 24
  Total Parameters: 1,259,751
  Test Accuracy   : 55.63%
  Test Macro F1   : 50.58%
  Test Weighted F1: 53.69%
  Test Loss       : 1.2577


---
## 03 · Environment & Reproducibility

Setting deterministic seeds, verifying CPU/hardware execution settings, and importing reusable modules from `ml/`.

In [2]:
# ── Section 03: Environment Setup & Reusable Imports ───────────────────────
import os
import sys
import time
from datetime import datetime, timezone
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Default backend; plots are explicitly saved & displayed
import matplotlib.pyplot as plt
import torch

# Ensure project root is in sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reusable modules from ml/
from ml.data.dataloader import load_yaml_config
from ml.models.builder import build_model_from_config, get_model_summary, print_model_summary
from ml.training.retrain_dataset import build_retraining_dataloaders
from ml.training.trainer import EmotionTrainer
from ml.training.model_comparator import (
    ModelComparator,
    load_checkpoint_into_model,
    evaluate_model_on_dataloader,
)
from ml.training.metrics import format_confusion_matrix_ascii, save_metrics
from ml.training.utils import set_seed, get_device
from ml.feedback.collector import FeedbackCollector

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 120,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

device, dev_info = get_device()
print("=" * 68)
print("  ENVIRONMENT & REPRODUCIBILITY INFO")
print("=" * 68)
print(f"  Python Version     : {sys.version.split()[0]}")
print(f"  PyTorch Version    : {torch.__version__}")
print(f"  Compute Device     : {dev_info.get('device', 'cpu')} ({dev_info.get('device_name', 'CPU')})")
print(f"  PyTorch CPU Threads: {torch.get_num_threads()}")
print(f"  Project Root       : {PROJECT_ROOT}")
print("=" * 68)


  ENVIRONMENT & REPRODUCIBILITY INFO
  Python Version     : 3.11.9
  PyTorch Version    : 2.14.0+cpu
  Compute Device     : cpu (CPU)
  PyTorch CPU Threads: 10
  Project Root       : c:\Users\moham\OneDrive\Desktop\AI projects\AI Emotion Detection


---
## 04 · Dataset Overview (FER2013 Base)

Inspects the base FER2013 dataset splits, image specifications, and class distribution.

> **Dataset Policy**: The raw `Data(FER2013)/` folder is strictly treated as read-only. The test split is untouched across all experiments to guarantee benchmark comparability.

In [3]:
# ── Section 04: FER2013 Dataset Inspection ─────────────────────────────────
base_config = load_yaml_config(PROJECT_ROOT / "configs" / "config.yaml")
train_dir = PROJECT_ROOT / base_config["dataset"]["train_split_dir"]
test_dir = PROJECT_ROOT / base_config["dataset"]["test_split_dir"]
class_names = base_config["classes"]["names"]

print("=" * 68)
print("  FER2013 BASE DATASET OVERVIEW")
print("=" * 68)
print(f"  Train Directory : {train_dir}")
print(f"  Test Directory  : {test_dir}")
print(f"  Image Dimensions: {base_config['dataset']['image_size']} (Grayscale)")
print(f"  Emotion Classes : {len(class_names)} classes {class_names}")

train_counts = {c: len(list((train_dir / c).glob("*.jpg"))) for c in class_names if (train_dir / c).exists()}
test_counts = {c: len(list((test_dir / c).glob("*.jpg"))) for c in class_names if (test_dir / c).exists()}

print("\n  Base Class Distribution:")
print(f"  {'Class':<12} {'Train Count':>14} {'Test Count':>14}")
print("  " + "-" * 42)
for c in class_names:
    print(f"  {c:<12} {train_counts.get(c, 0):>14,} {test_counts.get(c, 0):>14,}")
print("  " + "-" * 42)
print(f"  {'TOTAL':<12} {sum(train_counts.values()):>14,} {sum(test_counts.values()):>14,}")
print("=" * 68)


  FER2013 BASE DATASET OVERVIEW
  Train Directory : c:\Users\moham\OneDrive\Desktop\AI projects\AI Emotion Detection\Data(FER2013)\train
  Test Directory  : c:\Users\moham\OneDrive\Desktop\AI projects\AI Emotion Detection\Data(FER2013)\test
  Image Dimensions: [48, 48] (Grayscale)
  Emotion Classes : 7 classes ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

  Base Class Distribution:
  Class           Train Count     Test Count
  ------------------------------------------
  angry                 3,995            958
  disgust                 436            111
  fear                  4,097          1,024
  happy                 7,215          1,774
  neutral               4,965          1,233
  sad                   4,830          1,247
  surprise              3,171            831
  ------------------------------------------
  TOTAL                28,709          7,178


---
## 05 · Feedback Dataset Overview

Inspects collected user feedback records from `outputs/feedback/` using [`FeedbackCollector`](file:///c:/Users/moham/OneDrive/Desktop/AI%20projects/AI%20Emotion%20Detection/ml/feedback/collector.py).
Feedback records allow retraining with active human-in-the-loop corrections while preserving original datasets.

In [4]:
# ── Section 05: User Feedback Store Inspection ─────────────────────────────
feedback_dir = PROJECT_ROOT / "outputs" / "feedback"
collector = FeedbackCollector(feedback_dir=feedback_dir)
feedback_stats = collector.get_summary_statistics()

print("=" * 68)
print("  USER FEEDBACK DATASET OVERVIEW")
print("=" * 68)
print(f"  Feedback Directory : {feedback_dir}")
print(f"  Total Logged Records: {feedback_stats.get('total_records', 0)}")
print(f"  Breakdown by State : {feedback_stats.get('by_state', {})}")
print(f"  Predicted Emotions : {feedback_stats.get('by_predicted_emotion', {})}")
print(f"  Corrected Emotions : {feedback_stats.get('by_corrected_emotion', {})}")

dataset_dir = feedback_dir / "feedback_dataset"
if dataset_dir.exists():
    fb_train_images = list((dataset_dir / "train").glob("*/*.jpg")) if (dataset_dir / "train").exists() else []
    print(f"  Prepared Feedback Train Images : {len(fb_train_images)}")
print("=" * 68)


  USER FEEDBACK DATASET OVERVIEW
  Feedback Directory : c:\Users\moham\OneDrive\Desktop\AI projects\AI Emotion Detection\outputs\feedback
  Total Logged Records: 1
  Breakdown by State : {'correct': 1, 'incorrect': 0, 'uncertain': 0}
  Predicted Emotions : {'happy': 1}
  Corrected Emotions : {}
  Prepared Feedback Train Images : 1


---
## 06 · Experiment Configuration

This is the **single configuration cell** for the retraining experiment. Modify hyperparameters here before executing downstream training cells.

### Training Defaults
- **Architecture**: `ResidualEmotionCNN` (Residual blocks with skip connections)
- **Epochs**: `40` (with early stopping patience = 10 on Validation Macro F1)
- **Batch Size**: `64`
- **Learning Rate**: `0.001` with `CosineAnnealingLR`
- **Feedback Mixing**: `10%` feedback ratio cap (`0.10`)

In [5]:
# ── Section 06: User Experiment Configuration (Single Config Cell) ─────────
# *** EDIT HYPERPARAMETERS IN THIS CELL ***

# 1. Model Architecture
MODEL_ARCHITECTURE = "ResidualEmotionCNN"  # 'ResidualEmotionCNN' or 'BaselineEmotionCNN'

# 2. Training Hyperparameters
NUM_EPOCHS         = 40
BATCH_SIZE         = 64
LEARNING_RATE      = 0.001
WEIGHT_DECAY       = 0.0001
LABEL_SMOOTHING    = 0.05
RANDOM_SEED        = 42
EARLY_STOPPING     = True
EARLY_STOPPING_PATIENCE = 10
PRIMARY_METRIC     = "macro_f1"

# 3. Feedback Retraining Settings (Phase 11)
USE_FEEDBACK_DATA  = True          # Mix FER2013 with validated user feedback
FEEDBACK_RATIO     = 0.10          # Maximum fraction of feedback vs base train (10%)
FEEDBACK_WEIGHT    = 1.0           # Oversampling weight for feedback samples
FEEDBACK_DIR       = PROJECT_ROOT / "outputs" / "feedback"

# 4. Imbalance & Optimization Strategy
USE_WEIGHTED_SAMPLER    = True
USE_CLASS_WEIGHTED_LOSS = False     # False when WeightedRandomSampler is active
SCHEDULER_NAME          = "CosineAnnealingLR"
WARMUP_EPOCHS           = 2
MIN_LR                  = 1e-5

# 5. Baseline & Promotion Gate Settings
PRODUCTION_CKPT         = PROD_CKPT_PATH
BASELINE_VERSION        = "ResidualEmotionCNN-epoch24"
retrain_cfg             = base_config.get("retraining", {})
GATE_MIN_MACRO_F1_DELTA       = float(retrain_cfg.get("min_macro_f1_delta", 0.0))
GATE_MAX_ACCURACY_DROP        = float(retrain_cfg.get("max_accuracy_drop", 1.0))
GATE_MAX_CLASS_F1_DEGRADATION = float(retrain_cfg.get("max_class_f1_degradation", 3.0))

# 6. Isolated Experiment Output Paths
timestamp_str   = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_NAME = f"retrain_phase11_{timestamp_str}"
EXPERIMENT_DIR  = PROJECT_ROOT / "outputs" / "experiments" / EXPERIMENT_NAME
CHECKPOINT_DIR  = EXPERIMENT_DIR / "checkpoints"
METRICS_DIR     = EXPERIMENT_DIR / "metrics"
PLOTS_DIR       = EXPERIMENT_DIR / "plots"

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Assemble full experiment config dict
exp_config = {
    "project":  {"name": "FaceSense AI", "seed": RANDOM_SEED},
    "dataset":  {
        "name": "FER2013",
        "train_split_dir": str(PROJECT_ROOT / "Data(FER2013)" / "train"),
        "test_split_dir":  str(PROJECT_ROOT / "Data(FER2013)" / "test"),
        "image_size": [48, 48], "channels": 1,
        "val_split_ratio": 0.15, "seed": RANDOM_SEED,
    },
    "classes": {
        "num_classes": 7,
        "names":   ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"],
        "mapping": {"angry": 0, "disgust": 1, "fear": 2, "happy": 3,
                     "neutral": 4, "sad": 5, "surprise": 6},
    },
    "model": {
        "architecture": MODEL_ARCHITECTURE,
        "channel_list": [32, 64, 128, 256], "fc_dim": 128,
        "conv_dropout": 0.1, "fc_dropout": 0.4,
    },
    "augmentation": {
        "horizontal_flip_prob": 0.5, "rotation_degrees": 10,
        "crop_scale": [0.85, 1.0],
    },
    "dataloader": {
        "batch_size": BATCH_SIZE,
        "num_workers": 0,          # 0 = main process loading; safe & reliable on Windows
        "pin_memory": False, "persistent_workers": False, "prefetch_factor": None,
        "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
    },
    "cpu": {"num_threads": 8, "use_compile": False, "use_bf16_autocast": False},
    "training": {
        "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "optimizer": "adamw", "loss": "cross_entropy",
        "label_smoothing": LABEL_SMOOTHING,
        "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
        "use_class_weighted_loss": USE_CLASS_WEIGHTED_LOSS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE if EARLY_STOPPING else 9999,
        "scheduler": {
            "name": SCHEDULER_NAME, "warmup_epochs": WARMUP_EPOCHS,
            "warmup_start_factor": 0.1, "min_lr": MIN_LR,
            "mode": "max", "factor": 0.5, "patience": 3,
        },
    },
    "paths": {
        "checkpoints_dir":      str(CHECKPOINT_DIR),
        "best_checkpoint_path": str(CHECKPOINT_DIR / "candidate_best_model.pt"),
        "outputs_metrics_dir":  str(METRICS_DIR),
        "outputs_plots_dir":    str(PLOTS_DIR),
        "outputs_feedback_dir": str(FEEDBACK_DIR),
    },
}

set_seed(RANDOM_SEED)

print("=" * 68)
print(f"  EXPERIMENT CONFIGURATION READY")
print("=" * 68)
print(f"  Experiment Name   : {EXPERIMENT_NAME}")
print(f"  Architecture      : {MODEL_ARCHITECTURE}")
print(f"  Max Epochs        : {NUM_EPOCHS}")
print(f"  Batch Size        : {BATCH_SIZE}")
print(f"  Learning Rate     : {LEARNING_RATE}")
print(f"  Weight Decay      : {WEIGHT_DECAY}")
print(f"  Early Stopping    : {EARLY_STOPPING} (patience={EARLY_STOPPING_PATIENCE})")
print(f"  Primary Metric    : {PRIMARY_METRIC}")
print(f"  Use Feedback Data : {USE_FEEDBACK_DATA} (ratio={FEEDBACK_RATIO})")
print(f"  Output Directory  : {EXPERIMENT_DIR}")
print("=" * 68)


  EXPERIMENT CONFIGURATION READY
  Experiment Name   : retrain_phase11_20260910_151005
  Architecture      : ResidualEmotionCNN
  Max Epochs        : 40
  Batch Size        : 64
  Learning Rate     : 0.001
  Weight Decay      : 0.0001
  Early Stopping    : True (patience=10)
  Primary Metric    : macro_f1
  Use Feedback Data : True (ratio=0.1)
  Output Directory  : c:\Users\moham\OneDrive\Desktop\AI projects\AI Emotion Detection\outputs\experiments\retrain_phase11_20260910_151005


---
## 07 · Data Pipeline & Effective Splits

Builds stratified DataLoaders with optional feedback blending via [`build_retraining_dataloaders`](file:///c:/Users/moham/OneDrive/Desktop/AI%20projects/AI%20Emotion%20Detection/ml/training/retrain_dataset.py).
Transparently displays sample counts for each split to confirm dataset integrity.

In [6]:
# ── Section 07: DataPipeline Construction & Size Validation ────────────────
fb_path = FEEDBACK_DIR if USE_FEEDBACK_DATA else None
train_loader, val_loader, test_loader, data_meta = build_retraining_dataloaders(
    config=exp_config,
    feedback_dir=fb_path,
    feedback_sampling_ratio=FEEDBACK_RATIO if USE_FEEDBACK_DATA else 0.0,
    feedback_weight=FEEDBACK_WEIGHT if USE_FEEDBACK_DATA else 0.0,
)

CLASS_NAMES = data_meta["class_names"]

print("=" * 68)
print("  DATA PIPELINE SUMMARY (Effective Dataset Splits)")
print("=" * 68)
print(f"  Original FER2013 train : {data_meta['base_train_count']:>8,}")
print(f"  Feedback train         : {data_meta['feedback_samples_used']:>8,}")
print(f"  Combined train         : {data_meta['total_train_count']:>8,}")
print(f"  Validation             : {data_meta['val_count']:>8,}")
print(f"  Test (untouched)       : {data_meta['test_count']:>8,}")
print("-" * 68)
print(f"  Source Distribution    : {data_meta['source_distribution']}")
print(f"  Effective FB Weight    : {data_meta['effective_feedback_weight']}")
print(f"  Weighted Sampler Active: {USE_WEIGHTED_SAMPLER}")
print("\n  Combined Training Set Class Counts:")
for cls_name, cnt in data_meta['class_counts_train'].items():
    print(f"    {cls_name:<10}: {cnt:>6,} samples")
print("=" * 68)


  DATA PIPELINE SUMMARY (Effective Dataset Splits)
  Original FER2013 train :   24,402
  Feedback train         :        1
  Combined train         :   24,403
  Validation             :    4,307
  Test (untouched)       :    7,178
--------------------------------------------------------------------
  Source Distribution    : {'base': 24402, 'feedback': 1}
  Effective FB Weight    : 1.0
  Weighted Sampler Active: True

  Combined Training Set Class Counts:
    angry     :  3,396 samples
    disgust   :    371 samples
    fear      :  3,482 samples
    happy     :  6,134 samples
    neutral   :  4,220 samples
    sad       :  4,105 samples
    surprise  :  2,695 samples


---
## 08 · Model Configuration

Instantiates the candidate neural network using [`build_model_from_config`](file:///c:/Users/moham/OneDrive/Desktop/AI%20projects/AI%20Emotion%20Detection/ml/models/builder.py) and verifies parameter counts and tensor flow dimensions.

In [7]:
# ── Section 08: Candidate Model Instantiation ──────────────────────────────
candidate_model = build_model_from_config(exp_config)
print_model_summary(candidate_model, input_size=(1, 1, 48, 48))


Model Summary: ResidualEmotionCNN
Input Shape:            [1, 1, 48, 48]
Output Shape:           [1, 7]
Trainable Parameters:   1,259,751
Non-Trainable Params:   0
Total Parameters:       1,259,751
Model Size (FP32):      4.81 MB


---
## 09 · Training Execution

### Experiment Hypothesis
The current production model (`ResidualEmotionCNN-epoch24`) achieves **55.63% test accuracy** and **50.58% Macro F1**.
We are testing whether controlled feedback retraining with `CosineAnnealingLR` and `WeightedRandomSampler` can improve generalization without degrading performance on difficult or minor classes (such as `disgust` and `fear`).

> **TRANSPARENT EXECUTION**: The cell below executes [`EmotionTrainer`](file:///c:/Users/moham/OneDrive/Desktop/AI%20projects/AI%20Emotion%20Detection/ml/training/trainer.py) interactively inside the notebook, streaming epoch-by-epoch loss, accuracy, and Validation Macro F1 progress. Run this cell manually when you are ready to start training.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 09 · TRAINING CELL — Run this cell manually to start training
# ═══════════════════════════════════════════════════════════════════════════
t_train_start = time.time()

trainer = EmotionTrainer(
    model=candidate_model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    config=exp_config,
    class_names=CLASS_NAMES,
)

# Override paths to isolated experiment directory
trainer.checkpoints_dir      = CHECKPOINT_DIR
trainer.best_checkpoint_path = CHECKPOINT_DIR / "candidate_best_model.pt"
trainer.metrics_dir          = METRICS_DIR
trainer.plots_dir            = PLOTS_DIR

train_results = trainer.train(max_epochs=NUM_EPOCHS)

history          = train_results["history"]
best_epoch       = trainer.early_stopping.best_epoch
best_val_macro_f1= trainer.early_stopping.best_score
epochs_completed = len(history["epoch"])
stopped_early    = trainer.early_stopping.early_stop

CANDIDATE_CKPT   = CHECKPOINT_DIR / "candidate_best_model.pt"
CAND_VERSION     = f"{MODEL_ARCHITECTURE}-Candidate-epoch{best_epoch}"

train_elapsed_sec = time.time() - t_train_start
print("=" * 68)
print("  TRAINING RUN COMPLETED")
print("=" * 68)
print(f"  Total Elapsed Time  : {train_elapsed_sec/60:.2f} minutes ({train_elapsed_sec:.1f}s)")
print(f"  Epochs Completed    : {epochs_completed} / {NUM_EPOCHS}")
print(f"  Best Epoch          : {best_epoch}")
print(f"  Best Val Macro F1   : {best_val_macro_f1:.2f}%")
print(f"  Early Stopping Fired: {stopped_early}")
print(f"  Saved Checkpoint    : {CANDIDATE_CKPT}")
print("=" * 68)


FaceSense AI Training Pipeline
Device: {'device': 'cpu', 'cpu_threads': 12, 'torch_threads': 8}
Architecture: ResidualEmotionCNN
Compiled: False  |  BF16 Autocast: False
CPU Threads: 8
Total Epochs: 40 | Batch Size: 64
Primary Selection Metric: Validation Macro F1
Epoch [01/40] Train Loss: 5.8209 | Train Acc: 14.69% | Val Loss: 2.4002 | Val Acc: 17.46% | Val Macro F1: 15.41% | LR: 1.0e-04 | Time: 449.6s [*BEST*]
Epoch [02/40] Train Loss: 3.4272 | Train Acc: 15.60% | Val Loss: 1.9743 | Val Acc: 10.61% | Val Macro F1: 9.93% | LR: 5.5e-04 | Time: 513.3s
Epoch [03/40] Train Loss: 2.5252 | Train Acc: 15.94% | Val Loss: 1.9441 | Val Acc: 12.86% | Val Macro F1: 7.27% | LR: 1.0e-03 | Time: 477.4s
Epoch [04/40] Train Loss: 2.3230 | Train Acc: 16.64% | Val Loss: 1.9205 | Val Acc: 18.60% | Val Macro F1: 14.50% | LR: 1.0e-03 | Time: 483.9s
Epoch [05/40] Train Loss: 2.2169 | Train Acc: 18.11% | Val Loss: 1.9163 | Val Acc: 19.13% | Val Macro F1: 12.98% | LR: 9.9e-04 | Time: 468.4s
Epoch [06/40] Trai

---
## 10 · Training Curves

Plots training vs validation loss, accuracy, Validation Macro F1, and learning rate progression across all completed epochs.

In [ ]:
# ── Section 10: Training Curves Visualization ──────────────────────────────
epochs_axis = history["epoch"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Loss Curves
axes[0, 0].plot(epochs_axis, history["train_loss"], marker="o", color="#e74c3c", linewidth=2, markersize=3, label="Train Loss")
axes[0, 0].plot(epochs_axis, history["val_loss"], marker="s", color="#3498db", linewidth=2, markersize=3, label="Val Loss")
axes[0, 0].axvline(best_epoch, color="#c0392b", linestyle=":", label=f"Best Epoch {best_epoch}")
axes[0, 0].set_title("Cross Entropy Loss", fontweight="bold")
axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend(fontsize=9)

# 2. Accuracy Curves
axes[0, 1].plot(epochs_axis, history["train_acc"], marker="o", color="#e74c3c", linewidth=2, markersize=3, label="Train Acc")
axes[0, 1].plot(epochs_axis, history["val_acc"], marker="s", color="#3498db", linewidth=2, markersize=3, label="Val Acc")
axes[0, 1].axvline(best_epoch, color="#c0392b", linestyle=":")
axes[0, 1].set_title("Top-1 Accuracy (%)", fontweight="bold")
axes[0, 1].set_xlabel("Epoch"); axes[0, 1].set_ylabel("Accuracy (%)")
axes[0, 1].legend(fontsize=9)

# 3. Validation Macro F1
axes[1, 0].plot(epochs_axis, history["val_macro_f1"], marker="o", color="#2ecc71", linewidth=2, markersize=3, label="Val Macro F1")
axes[1, 0].axvline(best_epoch, color="#c0392b", linestyle=":", label=f"Best Epoch ({best_val_macro_f1:.2f}%)")
axes[1, 0].scatter([best_epoch], [best_val_macro_f1], color="#c0392b", s=90, zorder=5)
axes[1, 0].set_title("Primary Optimization Metric: Validation Macro F1", fontweight="bold")
axes[1, 0].set_xlabel("Epoch"); axes[1, 0].set_ylabel("Macro F1 (%)")
axes[1, 0].legend(fontsize=9)

# 4. Learning Rate Schedule
axes[1, 1].semilogy(epochs_axis, history["lr"], marker="d", color="#9b59b6", linewidth=2, markersize=4, label="Learning Rate")
axes[1, 1].set_title(f"LR Schedule ({SCHEDULER_NAME})", fontweight="bold")
axes[1, 1].set_xlabel("Epoch"); axes[1, 1].set_ylabel("LR (log scale)")
axes[1, 1].legend(fontsize=9)

fig.suptitle(f"{MODEL_ARCHITECTURE} Retraining Curves — {EXPERIMENT_NAME}", fontsize=13, fontweight="bold", y=0.99)
plt.tight_layout()
curves_path = PLOTS_DIR / "training_curves.png"
plt.savefig(curves_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Training curves plot saved to: {curves_path}")


---
## 11 · Best Epoch Analysis

Evaluates the convergence trajectory at the selected best checkpoint and generates automated observations from empirical metrics.

In [ ]:
# ── Section 11: Best Epoch Metric Breakdown & Dynamic Observation ──────────
best_idx = best_epoch - 1
best_train_loss = history["train_loss"][best_idx]
best_val_loss   = history["val_loss"][best_idx]
best_train_acc  = history["train_acc"][best_idx]
best_val_acc    = history["val_acc"][best_idx]
best_lr         = history["lr"][best_idx]

print("=" * 68)
print(f"  BEST CHECKPOINT METRIC SUMMARY (Epoch {best_epoch})")
print("=" * 68)
print(f"  Training Loss       : {best_train_loss:.4f}")
print(f"  Validation Loss     : {best_val_loss:.4f}")
print(f"  Training Accuracy   : {best_train_acc:.2f}%")
print(f"  Validation Accuracy : {best_val_acc:.2f}%")
print(f"  Validation Macro F1 : {best_val_macro_f1:.2f}%")
print(f"  Effective LR        : {best_lr:.6f}")
print("=" * 68)

# Dynamic observation generation
still_improving_at_limit = (
    not stopped_early
    and epochs_completed == NUM_EPOCHS
    and len(history["val_macro_f1"]) >= 3
    and history["val_macro_f1"][-1] >= history["val_macro_f1"][-3]
)

print("\n### Observation")
print(f"- Validation Macro F1 peaked at epoch {best_epoch} ({best_val_macro_f1:.2f}%).")
if stopped_early:
    print(f"- Early stopping triggered after {EARLY_STOPPING_PATIENCE} epochs without improvement beyond epoch {best_epoch}.")
elif still_improving_at_limit:
    print(f"- Validation metric was still trending upwards near epoch {NUM_EPOCHS}. A higher epoch limit (e.g. 50) may yield further gains.")
else:
    print(f"- Completed all {NUM_EPOCHS} epochs stably with best metric captured at epoch {best_epoch}.")


---
## 12 · Test Evaluation (Untouched FER2013 Test Set)

### Why Macro F1?
The FER2013 dataset has significant class imbalance (e.g. `disgust` has only 54 test samples while `happy` has 1,774).
Standard accuracy can be artificially inflated by performing well only on dominant classes.
**Macro F1 gives equal importance to every emotion category**, ensuring that improvements are genuine across all facial expressions.

Here we load the saved candidate checkpoint and evaluate it on the untouched test split.

In [ ]:
# ── Section 12: Independent Test Set Evaluation ────────────────────────────
assert CANDIDATE_CKPT.exists(), f"Candidate checkpoint not found at: {CANDIDATE_CKPT}"

cand_eval_model, cand_ckpt_dict = load_checkpoint_into_model(CANDIDATE_CKPT, exp_config, device)

t_eval_start = time.time()
candidate_test_loss, candidate_metrics = evaluate_model_on_dataloader(
    model=cand_eval_model,
    dataloader=test_loader,
    device=device,
    class_names=CLASS_NAMES,
)
eval_elapsed = time.time() - t_eval_start

print("=" * 68)
print(f"  CANDIDATE TEST RESULTS ({CAND_VERSION})")
print("=" * 68)
print(f"  Evaluation Time   : {eval_elapsed:.2f}s")
print(f"  Test Loss         : {candidate_test_loss:.4f}")
print(f"  Test Accuracy     : {candidate_metrics['accuracy']:.2f}%")
print(f"  Test Macro F1     : {candidate_metrics['macro_f1']:.2f}%")
print(f"  Test Weighted F1  : {candidate_metrics['weighted_f1']:.2f}%")
print("=" * 68)
print("\n  Classification Report:")
print(candidate_metrics["classification_report"])

# Save candidate metrics JSON
save_metrics(
    {**candidate_metrics, "test_loss": candidate_test_loss, "version": CAND_VERSION},
    output_dir=METRICS_DIR,
    prefix="candidate_test_metrics",
)
print(f"  Metrics saved to: {METRICS_DIR / 'candidate_test_metrics.json'}")


---
## 13 · Per-Class Performance Analysis

Detailed breakdown of precision, recall, and F1 score per emotion category.

In [ ]:
# ── Section 13: Per-Class Metric Visualizations ────────────────────────────
per_class = candidate_metrics["per_class"]
precisions = [per_class[c]["precision"] for c in CLASS_NAMES]
recalls    = [per_class[c]["recall"] for c in CLASS_NAMES]
f1_scores  = [per_class[c]["f1_score"] for c in CLASS_NAMES]
supports   = [per_class[c]["support"] for c in CLASS_NAMES]

x = np.arange(len(CLASS_NAMES))
w = 0.26
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

b1 = ax1.bar(x - w, precisions, w, label="Precision", color="#3498db", edgecolor="white")
b2 = ax1.bar(x,     recalls,    w, label="Recall",    color="#e67e22", edgecolor="white")
b3 = ax1.bar(x + w, f1_scores,  w, label="F1-Score",  color="#2ecc71", edgecolor="white")
ax1.set(title=f"Per-Class Metrics — {CAND_VERSION}", xlabel="Class", ylabel="Score (%)", xticks=x)
ax1.set_xticklabels([c.capitalize() for c in CLASS_NAMES], rotation=30)
ax1.legend(fontsize=9)
for bar in list(b1) + list(b2) + list(b3):
    h = bar.get_height()
    if h > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, h + 0.5, f"{h:.1f}", ha="center", va="bottom", fontsize=7)

bar_colors = ["#e74c3c" if f < 25 else "#f39c12" if f < 50 else "#27ae60" for f in f1_scores]
ax2.barh([c.capitalize() for c in CLASS_NAMES], f1_scores, color=bar_colors, height=0.55)
ax2.set(title="Per-Class F1 Score Ranking", xlabel="F1-Score (%)")
for i, (f1_val, sup) in enumerate(zip(f1_scores, supports)):
    ax2.text(f1_val + 0.5, i, f"{f1_val:.2f}% (n={sup})", va="center", fontsize=8)

plt.tight_layout()
pc_chart_path = PLOTS_DIR / "candidate_per_class.png"
plt.savefig(pc_chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Per-class chart saved: {pc_chart_path}")


---
## 14 · Confusion Matrix (Candidate Model)

Displays raw count and normalized confusion matrices to analyze class-to-class confusion patterns.

In [ ]:
# ── Section 14: Candidate Confusion Matrix ─────────────────────────────────
cm = np.array(candidate_metrics["confusion_matrix"])
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis].clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, data, fmt, title in [
    (axes[0], cm,      ".0f", f"Confusion Matrix (Raw Counts) — {CAND_VERSION}"),
    (axes[1], cm_norm, ".2f", f"Confusion Matrix (Normalized) — {CAND_VERSION}"),
]:
    im = ax.imshow(data, interpolation="nearest", cmap="Blues", vmin=0, vmax=(1.0 if fmt == ".2f" else data.max()))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(xticks=np.arange(len(CLASS_NAMES)), yticks=np.arange(len(CLASS_NAMES)),
           xticklabels=[c.capitalize() for c in CLASS_NAMES],
           yticklabels=[c.capitalize() for c in CLASS_NAMES],
           xlabel="Predicted Emotion", ylabel="True Emotion", title=title)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=9)
    thresh = data.max() / 2.0
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            ax.text(j, i, format(data[i, j], fmt), ha="center", va="center",
                    fontsize=8, color="white" if data[i, j] > thresh else "black")

plt.tight_layout()
cm_chart_path = PLOTS_DIR / "candidate_confusion_matrix.png"
plt.savefig(cm_chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Confusion matrix saved: {cm_chart_path}")


---
## 15 · Production Baseline vs Candidate Comparison

Evaluates the current production baseline on the exact same test loader and generates side-by-side comparison tables and delta charts.

In [ ]:
# ── Section 15: Baseline vs Candidate Comparative Analysis ─────────────────
# 1. Evaluate baseline model on test loader
base_model, base_ckpt_meta = load_checkpoint_into_model(PRODUCTION_CKPT, exp_config, device)
base_loss, baseline_metrics = evaluate_model_on_dataloader(
    model=base_model,
    dataloader=test_loader,
    device=device,
    class_names=CLASS_NAMES,
)

print("=" * 72)
print("  PRODUCTION BASELINE VS CANDIDATE COMPARISON")
print("=" * 72)
print(f"  {'Metric':<22} {'Production':>14} {'Candidate':>14} {'Delta':>12} {'Status':>8}")
print("-" * 72)

cmp_rows = [
    ("Accuracy (%)",    baseline_metrics['accuracy'],    candidate_metrics['accuracy']),
    ("Macro F1 (%)",    baseline_metrics['macro_f1'],    candidate_metrics['macro_f1']),
    ("Weighted F1 (%)", baseline_metrics['weighted_f1'], candidate_metrics['weighted_f1']),
    ("Test Loss",       base_loss,                       candidate_test_loss),
]
for label, b_val, c_val in cmp_rows:
    delta = c_val - b_val
    flag = "(+)" if delta > 0 else "(-)" if delta < 0 else "(=)"
    fmt = ".4f" if "Loss" in label else ".2f"
    print(f"  {label:<22} {b_val:>14{fmt}} {c_val:>14{fmt}} {delta:>+11{fmt}} {flag:>8}")

print("\n  PER-CLASS F1 COMPARISON")
print("-" * 72)
print(f"  {'Class':<12} {'Baseline F1':>14} {'Candidate F1':>14} {'Delta':>12} {'Status':>8}")
print("-" * 72)
improved_classes = []
degraded_classes = []
for cls_name in CLASS_NAMES:
    b_f1 = baseline_metrics['per_class'][cls_name]['f1_score']
    c_f1 = candidate_metrics['per_class'][cls_name]['f1_score']
    d_f1 = c_f1 - b_f1
    if d_f1 > 0:
        improved_classes.append((cls_name, d_f1))
        status_str = "[+]"
    elif d_f1 < 0:
        degraded_classes.append((cls_name, d_f1))
        status_str = "[-]"
    else:
        status_str = "[=]"
    print(f"  {cls_name:<12} {b_f1:>13.2f}% {c_f1:>13.2f}% {d_f1:>+11.2f}% {status_str:>8}")
print("=" * 72)

# Side-by-side per-class F1 bar chart
b_f1s = [baseline_metrics['per_class'][c]['f1_score'] for c in CLASS_NAMES]
c_f1s = [candidate_metrics['per_class'][c]['f1_score'] for c in CLASS_NAMES]
x_arr = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x_arr - width/2, b_f1s, width, label=f"Baseline ({BASELINE_VERSION})", color="#3498db", alpha=0.85)
ax.bar(x_arr + width/2, c_f1s, width, label=f"Candidate ({CAND_VERSION})", color="#2ecc71", alpha=0.85)
for i, (b, c) in enumerate(zip(b_f1s, c_f1s)):
    d = c - b
    col = "#27ae60" if d >= 0 else "#e74c3c"
    ax.text(i, max(b, c) + 1.5, f"{d:+.1f}%", ha="center", fontsize=8, color=col, fontweight="bold")
ax.set(xticks=x_arr, xlabel="Emotion Class", ylabel="F1 Score (%)", title="Per-Class F1: Baseline vs Candidate", ylim=(0, 100))
ax.set_xticklabels([c.capitalize() for c in CLASS_NAMES], rotation=30)
ax.legend(fontsize=9)
plt.tight_layout()
cmp_chart_path = PLOTS_DIR / "per_class_f1_comparison.png"
plt.savefig(cmp_chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Comparison chart saved: {cmp_chart_path}")


---
## 16 · Promotion Gate Decision

Uses [`ModelComparator`](file:///c:/Users/moham/OneDrive/Desktop/AI%20projects/AI%20Emotion%20Detection/ml/training/model_comparator.py) from `ml/training/` to evaluate safety rules and decide whether the candidate model is promoted.

### Promotion Rules
1. **Macro F1 Improvement**: Candidate Macro F1 must be $\ge$ Baseline Macro F1 $+ \Delta$ (min delta = `0.0%`).
2. **Accuracy Drop Bound**: Candidate accuracy drop must be $\le 1.0%$.
3. **Per-Class Regression Bound**: No individual emotion class F1 may degrade by more than `3.0%`.

> The production model (`ml/models/checkpoints/final/best_model.pt`) is **never modified** unless all criteria are satisfied.

In [ ]:
# ── Section 16: Safe Promotion Gate Execution ──────────────────────────────
comparator = ModelComparator(
    config_path=PROJECT_ROOT / "configs" / "config.yaml",
    min_macro_f1_delta=GATE_MIN_MACRO_F1_DELTA,
    max_accuracy_drop=GATE_MAX_ACCURACY_DROP,
    max_class_f1_degradation=GATE_MAX_CLASS_F1_DEGRADATION,
    device=device,
)

comparison_report = comparator.compare_models(
    baseline_checkpoint_path=PRODUCTION_CKPT,
    candidate_checkpoint_path=CANDIDATE_CKPT,
    baseline_version_name=BASELINE_VERSION,
    candidate_version_name=CAND_VERSION,
    test_loader=test_loader,
)

decision = comparison_report["decision"]

print("=" * 68)
print("  PROMOTION GATE EVALUATION REPORT")
print("=" * 68)
print(f"  Gate Criteria:")
print(f"    1. min_macro_f1_delta       >= {GATE_MIN_MACRO_F1_DELTA:+.2f}%")
print(f"    2. max_accuracy_drop        <= {GATE_MAX_ACCURACY_DROP:.2f}%")
print(f"    3. max_class_f1_degradation <= {GATE_MAX_CLASS_F1_DEGRADATION:.2f}%")
print("-" * 68)

diffs = comparison_report["differences"]
print(f"  Macro F1 Delta : {diffs['macro_f1_diff']:+.2f}%")
print(f"  Accuracy Delta : {diffs['accuracy_diff']:+.2f}%")
print(f"  Loss Delta     : {diffs['loss_diff']:+.4f}")

print("\n  Per-Class Safety Gate Check:")
for cls_name in CLASS_NAMES:
    p_diff = diffs["per_class_diff"].get(cls_name, {})
    b_f1 = p_diff.get("baseline_f1", 0.0)
    c_f1 = p_diff.get("candidate_f1", 0.0)
    d_f1 = p_diff.get("f1_diff", 0.0)
    gate_ok = "[PASS]" if d_f1 >= -GATE_MAX_CLASS_F1_DEGRADATION else "[FAIL]"
    print(f"    {cls_name:<10}  Baseline={b_f1:5.2f}%  Candidate={c_f1:5.2f}%  Delta={d_f1:+6.2f}%  {gate_ok}")

print("-" * 68)
if decision == "PROMOTE":
    print("  DECISION: PROMOTE")
    print("  The candidate model has satisfied all promotion gate criteria.")
else:
    print("  DECISION: REJECT")
    print("  Rejection Reasons:")
    for reason in comparison_report.get("rejection_reasons", []):
        print(f"    - {reason}")
print("=" * 68)

# Save comparison report
report_path = METRICS_DIR / "comparison_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(comparison_report, f, indent=2)
print(f"  Comparison report saved: {report_path}")


> **Apply Promotion**: The following cell applies promotion if and only if the decision was `PROMOTE`. If the decision was `REJECT`, it safely no-ops without altering production files.

In [ ]:
# ── Apply Promotion (Safe No-Op on REJECT) ─────────────────────────────────
promotion_result = comparator.apply_promotion(
    comparison_report=comparison_report,
    production_dir=PROD_CKPT_PATH.parent,
)

print("=" * 68)
print(f"  Promotion Status : {promotion_result['status']}")
print(f"  Message          : {promotion_result['message']}")
if promotion_result.get("backup_checkpoint"):
    print(f"  Previous Backup  : {promotion_result['backup_checkpoint']}")
if promotion_result.get("production_checkpoint"):
    print(f"  Production Path  : {promotion_result['production_checkpoint']}")
print("=" * 68)

prod_changed = promotion_result.get("promoted", False)


---
## 17 · Experiment Artifacts

Lists all artifacts created during this experiment run under `outputs/experiments/`.

In [ ]:
# ── Section 17: Experiment Artifacts Inventory ─────────────────────────────
print("=" * 68)
print(f"  EXPERIMENT ARTIFACTS INVENTORY ({EXPERIMENT_NAME})")
print("=" * 68)
for root_p, dirs, files in os.walk(EXPERIMENT_DIR):
    rel_root = Path(root_p).relative_to(PROJECT_ROOT)
    for fname in sorted(files):
        f_sz = (Path(root_p) / fname).stat().st_size
        print(f"  {rel_root / fname} ({f_sz:,} bytes)")
print("=" * 68)


---
## 18 · Final Experiment Summary & Automated Observations

Consolidates all configuration parameters, dataset stats, training metrics, baseline comparisons, and automated analytical observations into a reproducible experiment manifest (`experiment_manifest.json`).

In [ ]:
# ── Section 18: Compact Summary Table & Automated Insights ─────────────────
# Identify strongest & weakest classes dynamically from test metrics
sorted_classes = sorted(
    CLASS_NAMES,
    key=lambda c: candidate_metrics["per_class"][c]["f1_score"],
    reverse=True,
)
strongest_class = sorted_classes[0]
weakest_class   = sorted_classes[-1]

acc_delta = candidate_metrics["accuracy"] - baseline_metrics["accuracy"]
mf1_delta = candidate_metrics["macro_f1"] - baseline_metrics["macro_f1"]
wf1_delta = candidate_metrics["weighted_f1"] - baseline_metrics["weighted_f1"]
loss_delta = candidate_test_loss - base_loss

# Build reproducible experiment manifest dict
manifest = {
    "experiment_name":    EXPERIMENT_NAME,
    "timestamp_utc":      datetime.now(timezone.utc).isoformat(),
    "model_architecture": MODEL_ARCHITECTURE,
    "model_parameters":   get_model_summary(candidate_model, input_size=(1, 1, 48, 48)),
    "configuration": {
        "num_epochs":             NUM_EPOCHS,
        "batch_size":             BATCH_SIZE,
        "learning_rate":          LEARNING_RATE,
        "weight_decay":           WEIGHT_DECAY,
        "early_stopping":         EARLY_STOPPING,
        "early_stopping_patience":EARLY_STOPPING_PATIENCE,
        "primary_metric":         PRIMARY_METRIC,
        "use_feedback_data":      USE_FEEDBACK_DATA,
        "feedback_ratio":         FEEDBACK_RATIO,
        "feedback_weight":        FEEDBACK_WEIGHT,
        "scheduler_name":         SCHEDULER_NAME,
        "random_seed":            RANDOM_SEED,
    },
    "dataset": {
        "fer2013_base_train": data_meta["base_train_count"],
        "feedback_train":     data_meta["feedback_samples_used"],
        "total_train":        data_meta["total_train_count"],
        "validation":         data_meta["val_count"],
        "test_untouched":     data_meta["test_count"],
    },
    "training": {
        "epochs_completed":   epochs_completed,
        "best_epoch":         best_epoch,
        "best_val_macro_f1":  round(best_val_macro_f1, 4),
        "stopped_early":      stopped_early,
    },
    "baseline": {
        "version":     BASELINE_VERSION,
        "accuracy":    round(baseline_metrics["accuracy"], 4),
        "macro_f1":    round(baseline_metrics["macro_f1"], 4),
        "weighted_f1": round(baseline_metrics["weighted_f1"], 4),
        "test_loss":   round(base_loss, 6),
    },
    "candidate": {
        "version":     CAND_VERSION,
        "checkpoint":  str(CANDIDATE_CKPT),
        "accuracy":    round(candidate_metrics["accuracy"], 4),
        "macro_f1":    round(candidate_metrics["macro_f1"], 4),
        "weighted_f1": round(candidate_metrics["weighted_f1"], 4),
        "test_loss":   round(candidate_test_loss, 6),
    },
    "improvement": {
        "accuracy_delta":    round(acc_delta, 4),
        "macro_f1_delta":    round(mf1_delta, 4),
        "weighted_f1_delta": round(wf1_delta, 4),
        "loss_delta":        round(loss_delta, 6),
    },
    "promotion": {
        "decision":                   decision,
        "status":                     promotion_result.get("status"),
        "final_best_model_pt_changed": prod_changed,
    },
}

manifest_path = EXPERIMENT_DIR / "experiment_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("=" * 72)
print("  FINAL EXPERIMENT SUMMARY (Compact View)")
print("=" * 72)
print(f"  {'Category':<15} {'Details / Metrics'}")
print("-" * 72)
print(f"  {'Configuration':<15} {MODEL_ARCHITECTURE} | {NUM_EPOCHS} ep | lr={LEARNING_RATE} | fb_ratio={FEEDBACK_RATIO}")
print(f"  {'Dataset':<15} Train: {data_meta['total_train_count']:,} (Base: {data_meta['base_train_count']:,} + FB: {data_meta['feedback_samples_used']:,}) | Val: {data_meta['val_count']:,} | Test: {data_meta['test_count']:,}")
print(f"  {'Training':<15} Best Epoch {best_epoch}/{epochs_completed} | Val Macro F1: {best_val_macro_f1:.2f}% | Early Stop: {stopped_early}")
print(f"  {'Baseline':<15} Acc: {baseline_metrics['accuracy']:.2f}% | Macro F1: {baseline_metrics['macro_f1']:.2f}% | Loss: {base_loss:.4f}")
print(f"  {'Candidate':<15} Acc: {candidate_metrics['accuracy']:.2f}% | Macro F1: {candidate_metrics['macro_f1']:.2f}% | Loss: {candidate_test_loss:.4f}")
print(f"  {'Improvement':<15} Acc: {acc_delta:+.2f}% | Macro F1: {mf1_delta:+.2f}% | Loss: {loss_delta:+.4f}")
print(f"  {'Decision':<15} {decision} (best_model.pt updated: {prod_changed})")
print("=" * 72)

print("\n### Automated Experiment Observations")
print(f"- Best Epoch: {best_epoch} with Validation Macro F1 = {best_val_macro_f1:.2f}%")
print(f"- Strongest emotion class on test set: '{strongest_class}' (F1 = {candidate_metrics['per_class'][strongest_class]['f1_score']:.2f}%)")
print(f"- Weakest emotion class on test set  : '{weakest_class}' (F1 = {candidate_metrics['per_class'][weakest_class]['f1_score']:.2f}%)")

if improved_classes:
    improved_str = ', '.join([f'{c} ({d:+.2f}%)' for c, d in improved_classes])
    print(f"- Classes improved vs baseline: {improved_str}")
else:
    print("- No classes showed improvement over the baseline in this run.")

if degraded_classes:
    degraded_str = ', '.join([f'{c} ({d:+.2f}%)' for c, d in degraded_classes])
    print(f"- Classes degraded vs baseline: {degraded_str}")
else:
    print("- No classes degraded relative to the baseline.")

if still_improving_at_limit:
    print("- [Recommendation] Validation metric was still ascending at epoch limit. Consider running a 50-epoch experiment.")
print("=" * 72)
print(f"  Manifest exported to: {manifest_path}")
